# Heart Disease Prediction System

A binary classification ML task that predicts whether a patient **has heart disease (Yes)** or **does not (No)** from four simple inputs, using **Logistic Regression**. This notebook follows the same steps as the Titanic Passenger Survival Prediction notebook (Steps 1-10).

**Dataset:** Heart Disease dataset (UCI Machine Learning Repository) - https://archive.ics.uci.edu/dataset/45/heart+disease
We use the **Cleveland** file (`processed.cleveland.data`, 303 patients) from the downloaded zip, which is the standard, clean version of this dataset.

## Input and Output

Only **4 input attributes** are selected from the 13 available (task constraint).

| **Input Attribute** | **Meaning** | **Possible Values** |
| --- | --- | --- |
| **Gender** | Sex of the patient | Male, Female |
| **ChestPain** | Chest pain type | 1 - Typical Angina, 2 - Atypical Angina, 3 - Non-anginal Pain, 4 - Asymptomatic |
| **MaxHeartRate** | Maximum heart rate achieved (beats/min) | Number (about 70 to 210) |
| **ExerciseAngina** | Chest pain induced by exercise | Yes, No |

| **Output Attribute** | **Possible Values** |
| --- | --- |
| **HeartDisease** | Yes, No |

## Table of Content
- Step 1: Import Libraries
- Step 2: Load Sample Data
- Step 3: Understand and Pre-process Sample Data (3.1 Understand, 3.2 Pre-process, 3.3 Visualize)
- Step 4: Feature Extraction (select 4 inputs)
- Step 5: Label Encoding (5.1 Train encoders, 5.2 Encode output, 5.3 Encode input)
- Step 6: Training Phase (6.1 Split, 6.2 Input/Output split, 6.3 Train Logistic Regression, 6.4 Save model)
- Step 7: Testing Phase (7.1 Split, 7.2 Load model, 7.3 Predict, 7.4 Accuracy, 7.5 Precision/Recall/F1/Confusion Matrix)
- Step 8: Application Phase (user input -> prediction)
- Step 9: Feedback Phase
- Step 10: Deploy the Model to the Cloud (Streamlit Community Cloud)

**Folder needed:** put `processed.cleveland.data` (from the zip) in the same folder as this notebook.

### Step 1: Import Libraries

In [ ]:
"""
Purpose: import the libraries for data handling, preprocessing, model training,
evaluation and plotting.
"""
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

### Step 2: Load Sample Data

In [ ]:
"""
Purpose: load the Cleveland heart disease data from the zip's processed file.
The file has no header row, so the 14 column names are supplied here.
Missing values are written as '?' in the file, so they are read as NaN.
"""
column_names = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
                "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

raw_data = pd.read_csv("processed.cleveland.data", header=None,
                       names=column_names, na_values="?")

# Build a readable sample dataset with ONLY the 4 selected inputs + the output
chest_pain_names = {1: "1 - Typical Angina", 2: "2 - Atypical Angina",
                    3: "3 - Non-anginal Pain", 4: "4 - Asymptomatic"}

sample_data = pd.DataFrame({
    "Gender":         raw_data["sex"].map({1: "Male", 0: "Female"}),
    "ChestPain":      raw_data["cp"].astype(int).map(chest_pain_names),
    "MaxHeartRate":   raw_data["thalach"].astype(int),
    "ExerciseAngina": raw_data["exang"].map({1: "Yes", 0: "No"}),
    # original 'num' is 0 (no disease) or 1-4 (disease present) -> Yes/No
    "HeartDisease":   np.where(raw_data["num"] > 0, "Yes", "No"),
})

sample_data.to_csv("sample-data.csv", index=False)

print("\n\nSample Data (first 15 rows):")
print("============================\n")
print(sample_data.head(15))

### Step 3: Understand and Pre-process Sample Data
#### Step 3.1: Understand Sample Data

In [ ]:
print("\n\nAttributes in Sample Data:")
print("==========================\n")
print(sample_data.columns)

print("\n\nNumber of Instances in Sample Data:", sample_data["Gender"].count())
print("========================================\n")

print("Class distribution of the output (HeartDisease):")
print(sample_data["HeartDisease"].value_counts())

print("\nSummary of MaxHeartRate:")
print(sample_data["MaxHeartRate"].describe().round(1))

#### Step 3.2: Pre-process Sample Data
Check for missing values in the four selected attributes. (In the Cleveland file, the only missing values are in `ca` and `thal`, which we do not use.)

In [ ]:
print("Missing values per attribute:")
print(sample_data.isna().sum())

# Safety net: drop any rows with a missing value (expected: none)
before = len(sample_data)
sample_data = sample_data.dropna().reset_index(drop=True)
print("\nRows before:", before, "| Rows after:", len(sample_data))

#### Step 3.3: Visualize Heart Disease Rate by Gender and Chest Pain Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_df = sample_data.assign(Disease=(sample_data["HeartDisease"] == "Yes").astype(int))

by_gender = plot_df.groupby("Gender")["Disease"].mean()
sns.barplot(x=by_gender.index, y=by_gender.values, ax=axes[0],
            hue=by_gender.index, legend=False, palette="Blues_d")
axes[0].set_title("Heart Disease Rate by Gender")
axes[0].set_ylabel("Fraction with heart disease")
axes[0].set_ylim(0, 1)

by_cp = plot_df.groupby("ChestPain")["Disease"].mean()
sns.barplot(x=by_cp.index, y=by_cp.values, ax=axes[1],
            hue=by_cp.index, legend=False, palette="Oranges_d")
axes[1].set_title("Heart Disease Rate by Chest Pain Type")
axes[1].set_ylabel("Fraction with heart disease")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()
print("Observation: asymptomatic chest pain and male patients show a much higher heart disease rate.")

### Step 4: Feature Extraction
The task allows only **4 input attributes** out of 13. We compare a few candidate sets with 5-fold cross-validation using Logistic Regression and keep a set that is accurate and easy for a user to fill in.

In [ ]:
numeric_data = raw_data.dropna().copy()
numeric_data["target"] = (numeric_data["num"] > 0).astype(int)

candidate_sets = {
    "sex, cp, thalach, exang (chosen)": ["sex", "cp", "thalach", "exang"],
    "cp, thalach, oldpeak, exang":      ["cp", "thalach", "oldpeak", "exang"],
    "age, sex, cp, thalach":            ["age", "sex", "cp", "thalach"],
    "sex, cp, thalach, oldpeak":        ["sex", "cp", "thalach", "oldpeak"],
}

print("5-fold cross-validation accuracy of Logistic Regression:\n")
for name, cols in candidate_sets.items():
    score = cross_val_score(LogisticRegression(max_iter=1000),
                            numeric_data[cols], numeric_data["target"], cv=5).mean()
    print(f"  {name:36s} {score:.3f}")

print("\nChosen inputs: Gender (sex), ChestPain (cp), MaxHeartRate (thalach), ExerciseAngina (exang)")

### Step 5: Label Encoding the Sample Data
#### Step 5.1: Train the Label Encoder

In [ ]:
gender_label_encoder    = LabelEncoder().fit(["Male", "Female"])
chestpain_label_encoder = LabelEncoder().fit(list(chest_pain_names.values()))
exang_label_encoder     = LabelEncoder().fit(["Yes", "No"])
disease_label_encoder   = LabelEncoder().fit(["Yes", "No"])

print("Gender classes:        ", gender_label_encoder.classes_)
print("ChestPain classes:     ", chestpain_label_encoder.classes_)
print("ExerciseAngina classes:", exang_label_encoder.classes_)
print("HeartDisease classes:  ", disease_label_encoder.classes_, "-> No = 0, Yes = 1")

#### Step 5.2: Label Encode the Output

In [ ]:
sample_data_encoded = sample_data.copy()
sample_data_encoded["HeartDisease"] = disease_label_encoder.transform(sample_data["HeartDisease"])

print("HeartDisease after Label Encoding:")
print(pd.DataFrame({"HeartDisease": sample_data["HeartDisease"],
                    "encoded": sample_data_encoded["HeartDisease"]}).head(10))

#### Step 5.3: Label Encode the Input
`MaxHeartRate` is already numeric, so only the three categorical inputs are encoded.

In [ ]:
sample_data_encoded["Gender"]         = gender_label_encoder.transform(sample_data["Gender"])
sample_data_encoded["ChestPain"]      = chestpain_label_encoder.transform(sample_data["ChestPain"])
sample_data_encoded["ExerciseAngina"] = exang_label_encoder.transform(sample_data["ExerciseAngina"])

print("Sample Data after Label Encoding:")
print("=================================\n")
print(sample_data_encoded.head(15))
sample_data_encoded.to_csv("sample-data-encoded.csv", index=False)

### Step 6: Execute the Training Phase
#### Step 6.1: Splitting Sample Data into Training Data and Testing Data

In [ ]:
# 80% training, 20% testing. stratify keeps the Yes/No ratio the same in both parts.
training_data_encoded, testing_data_encoded = train_test_split(
    sample_data_encoded, test_size=0.2, random_state=0,
    stratify=sample_data_encoded["HeartDisease"])

training_data_encoded.to_csv("training-data-encoded.csv", index=False)
testing_data_encoded.to_csv("testing-data-encoded.csv", index=False)

print("Training Data shape:", training_data_encoded.shape)
print("Testing Data shape: ", testing_data_encoded.shape)

#### Step 6.2: Splitting Input Vectors and Outputs / Labels of Training Data

In [ ]:
input_vector_train = training_data_encoded.iloc[:, :-1]
output_label_train = training_data_encoded.iloc[:, -1]

print("Input vectors (train) shape:", input_vector_train.shape)
print("Output labels (train) shape:", output_label_train.shape)

#### Step 6.3: Train the Logistic Regression Model

In [ ]:
print("Training Logistic Regression on Training Data")
print("=============================================\n")

logistic_model = LogisticRegression(max_iter=1000, random_state=0)
logistic_model.fit(input_vector_train, np.ravel(output_label_train))

print(logistic_model)

print("\nLearned coefficients (positive = raises the chance of heart disease):")
for name, coef in zip(input_vector_train.columns, logistic_model.coef_[0]):
    print(f"  {name:15s} {coef:+.3f}")
print(f"  {'intercept':15s} {logistic_model.intercept_[0]:+.3f}")

#### Step 6.4: Save the Trained Model
The model and the label encoders are saved, because the app needs both to turn user input into the same numbers the model was trained on.

In [ ]:
pickle.dump(logistic_model, open("logistic_regression_model.pkl", "wb"))

encoders = {
    "gender": gender_label_encoder,
    "chestpain": chestpain_label_encoder,
    "exang": exang_label_encoder,
    "disease": disease_label_encoder,
}
pickle.dump(encoders, open("label_encoders.pkl", "wb"))

print("Saved: logistic_regression_model.pkl")
print("Saved: label_encoders.pkl")

### Step 7: Execute the Testing Phase
#### Step 7.1: Splitting Input Vectors and Outputs/Labels of Testing Data

In [ ]:
input_vector_test = testing_data_encoded.iloc[:, :-1]
output_label_test = testing_data_encoded.iloc[:, -1]

print("Input vectors (test) shape:", input_vector_test.shape)
print("Output labels (test) shape:", output_label_test.shape)

#### Step 7.2: Load the Saved Model

In [ ]:
model = pickle.load(open("logistic_regression_model.pkl", "rb"))
print("Model loaded from logistic_regression_model.pkl")

#### Step 7.3: Make Predictions with the Trained Model on Testing Data

In [ ]:
model_predictions = testing_data_encoded.copy()
model_predictions["Predictions"] = model.predict(input_vector_test)
model_predictions.to_csv("model-predictions.csv", index=False)

print("Predictions Returned by the Trained Model (first 15 test rows):")
print("===============================================================\n")
print(model_predictions.head(15))

#### Step 7.4: Calculate the Accuracy Score

In [ ]:
model_accuracy_score = accuracy_score(model_predictions["HeartDisease"],
                                      model_predictions["Predictions"])
print("Accuracy Score:", round(model_accuracy_score, 2))

#### Step 7.5: Precision, Recall, F1-score and Confusion Matrix

In [ ]:
print("Classification Report:")
print("======================\n")
print(classification_report(model_predictions["HeartDisease"],
                            model_predictions["Predictions"],
                            target_names=["No Heart Disease (0)", "Heart Disease (1)"]))

conf_matrix = confusion_matrix(model_predictions["HeartDisease"],
                               model_predictions["Predictions"])
print("Confusion Matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(conf_matrix,
                   index=["Actual: No", "Actual: Yes"],
                   columns=["Predicted: No", "Predicted: Yes"]))

plt.figure(figsize=(5, 4))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Logistic Regression")
plt.tight_layout()
plt.show()

### Step 8: Execute the Application Phase
#### Step 8.1: Take Input from User
A demo patient is pre-filled so the notebook runs top to bottom. To type your own values, replace the four lines with `input()` calls.

In [ ]:
gender_input         = "Male"                    # Male, Female
chestpain_input      = "4 - Asymptomatic"        # 1 - Typical Angina, 2 - Atypical Angina, 3 - Non-anginal Pain, 4 - Asymptomatic
maxheartrate_input   = 120                       # beats per minute
exerciseangina_input = "Yes"                     # Yes, No

print(f"Gender: {gender_input}, ChestPain: {chestpain_input}, "
      f"MaxHeartRate: {maxheartrate_input}, ExerciseAngina: {exerciseangina_input}")

#### Step 8.2: Convert User Input into Feature Vector

In [ ]:
user_input = pd.DataFrame({
    "Gender": [gender_input],
    "ChestPain": [chestpain_input],
    "MaxHeartRate": [maxheartrate_input],
    "ExerciseAngina": [exerciseangina_input],
})
print(user_input)

#### Step 8.3: Label Encoding of Feature Vector

In [ ]:
unseen_data_features = user_input.copy()
unseen_data_features["Gender"]         = gender_label_encoder.transform(user_input["Gender"])
unseen_data_features["ChestPain"]      = chestpain_label_encoder.transform(user_input["ChestPain"])
unseen_data_features["ExerciseAngina"] = exang_label_encoder.transform(user_input["ExerciseAngina"])
print(unseen_data_features)

#### Step 8.4: Load the Saved Model

In [ ]:
model = pickle.load(open("logistic_regression_model.pkl", "rb"))

#### Step 8.5: Model Prediction

In [ ]:
predicted_class = model.predict(unseen_data_features)[0]
probability_yes = model.predict_proba(unseen_data_features)[0][1]

prediction = "HEART DISEASE LIKELY" if predicted_class == 1 else "NO HEART DISEASE LIKELY"
print("** Prediction **:", prediction)
print(f"Estimated probability of heart disease: {probability_yes:.1%}")

### Step 9: Execute the Feedback Phase

**9.1 Feedback collected (example):**
- A domain expert (doctor) would confirm the model follows known medical patterns: asymptomatic chest pain, exercise-induced angina and a low maximum heart rate all raise the predicted risk.
- A tester noted that the model gives only a yes/no answer and the probability is more useful than the label alone.
- The model uses only 4 attributes of 13, so some patients with similar inputs but different blood tests or scan results cannot be told apart.

**9.2 Potential improvements:**
1. Add more inputs such as `oldpeak`, `ca`, `thal` and `age` (accuracy would likely rise).
2. Use all four UCI hospital files (Cleveland, Hungarian, Switzerland, VA) for more data.
3. Compare other models (Decision Tree, Random Forest, SVC) and use cross-validation.
4. Show the probability, not just the label, in the app (done in the deployed app).

**9.3 Improve the model:** retrain with the improvements, re-run Steps 6-7, and replace the deployed `.pkl` only if the new model performs better.

> **Important:** this is a student machine-learning project, not a medical device. It must not be used for real diagnosis.

### Step 10: Deploy the Model to the Cloud (Streamlit Community Cloud)

The project folder already contains everything needed:

| File | Purpose |
| --- | --- |
| `app.py` | The Streamlit web app |
| `requirements.txt` | Packages the cloud must install |
| `logistic_regression_model.pkl` | The trained model (created in Step 6.4) |
| `label_encoders.pkl` | The trained label encoders (created in Step 6.4) |
| `processed.cleveland.data` | Data file (the app retrains from it if the `.pkl` can't be loaded) |

**Steps**
1. Create a free account at https://github.com and make a new **Public** repository, for example `heart-disease-prediction`.
2. Upload all the files listed above to the repository (**Add file -> Upload files**), then commit.
3. Go to https://share.streamlit.io and sign in with GitHub.
4. Click **Create app -> Deploy a public app from GitHub**. Choose your repository, branch `main` and main file path `app.py`, then click **Deploy**.
5. If the build fails with a package error, open **Settings -> General -> Python version** and pick **3.11** or **3.12**, then reboot.
6. Copy the live `https://<name>.streamlit.app` link and put it, with your GitHub repo link, in your assignment submission.